In [8]:
from utilsforecast.preprocessing import fill_gaps
from tinyshift.stats import remove_leading_zeros, is_obsolete
from tinyshift.plot import stationarity_analysis, pami, residual_analysis, seasonal_decompose
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import kagglehub
from statsmodels.tsa.seasonal import MSTL
from statsforecast.models import SeasonalNaive, AutoETS
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from mlforecast import MLForecast
from utilsforecast.evaluation import evaluate
from utilsforecast.losses import rmse, mae, bias, cfe
from tinyshift.modelling import DMSTLWrapper, fourier_seasonality
from mlforecast.utils import PredictionIntervals
from tinyshift.series import (
    wape,
    pbias,
    score,
    forecast_instability,
    detect_seasonal_periods,
    extract_mstl_components,
    fva_rmae,
    mach,
    permutation_auto_mutual_information,
)

In [2]:
url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv'
df = pd.read_csv(url, parse_dates=['Month'])
df["unique_id"] = "1"
df.rename(columns={"Month": "ds", "Passengers": "y"}, inplace=True)
df = fill_gaps(df, freq="ME", end="per_serie", id_col="unique_id", time_col="ds")
df = df.groupby("unique_id")[df.columns].apply(remove_leading_zeros).reset_index(drop=True)
df = fourier_seasonality(df, "ds", seasonality=["monthly"])
days_obsoletes=180
obsolete_series = df.groupby("unique_id")[df.columns].apply(is_obsolete, days_obsoletes)
obsolote_ids = obsolete_series[obsolete_series].index.tolist()
assert len(obsolote_ids) == 0, f"Obsolete series found: {obsolote_ids}"

In [3]:
metrics = [rmse, mae, bias, wape, pbias, score, forecast_instability]

In [4]:
df.isnull().sum()

unique_id      0
ds             0
y              0
monthly_sin    0
monthly_cos    0
dtype: int64

In [ ]:
from pathlib import Path
from scipy.signal import find_peaks


def first_pami_local_minimum(values, max_tau=365, m=3, delay=1):
    values = np.asarray(values, dtype=float)
    max_valid_tau = len(values) - (m - 1) * delay - 1
    taus = np.arange(1, min(max_tau, max_valid_tau) + 1)
    pami_values = np.array([
        permutation_auto_mutual_information(
            values, tau=int(tau), m=m, delay=delay, normalize=True
        )
        for tau in taus
    ])
    minima, _ = find_peaks(-pami_values)
    position = minima[0] if len(minima) else int(np.argmin(pami_values))
    return int(taus[position]), float(pami_values[position])


# Online Retail II: mesmo painel diário e os mesmos 12 SKUs do demand_class.
dataset_dir = Path(kagglehub.dataset_download("mashlyn/online-retail-ii-uci"))
retail_files = list(dataset_dir.rglob("*.csv"))
raw_retail = pd.read_csv(retail_files[0], encoding="latin1")
price_col = "Price" if "Price" in raw_retail.columns else "UnitPrice"
retail = raw_retail[["StockCode", "InvoiceDate", "Quantity", price_col]].copy()
retail["InvoiceDate"] = pd.to_datetime(retail["InvoiceDate"], errors="coerce")
retail["StockCode"] = retail["StockCode"].astype(str)
retail = retail.loc[
    retail["InvoiceDate"].notna()
    & (retail["Quantity"] > 0)
    & (retail[price_col] > 0)
].copy()
retail["ds"] = retail["InvoiceDate"].dt.floor("D")

benchmark_skus = (
    retail.groupby("StockCode")["Quantity"]
    .sum()
    .nlargest(12)
    .index
    .tolist()
)
retail = retail[retail["StockCode"].isin(benchmark_skus)].copy()
dates = pd.date_range(retail["ds"].min(), retail["ds"].max(), freq="D")
benchmark_panel = pd.MultiIndex.from_product(
    [dates, benchmark_skus], names=["ds", "unique_id"]
).to_frame(index=False)
benchmark_values = (
    retail.groupby(["ds", "StockCode"])["Quantity"]
    .sum()
    .rename("y")
    .reset_index()
    .rename(columns={"StockCode": "unique_id"})
)
benchmark_panel = benchmark_panel.merge(
    benchmark_values, on=["ds", "unique_id"], how="left"
)
benchmark_panel["y"] = benchmark_panel["y"].fillna(0.0)

benchmark_cutoff = benchmark_panel["ds"].quantile(0.80)
benchmark_train = benchmark_panel[benchmark_panel["ds"] <= benchmark_cutoff].copy()
benchmark_test = benchmark_panel[benchmark_panel["ds"] > benchmark_cutoff].copy()
benchmark_horizon = 28

# Os periodos sazonais continuam vindo do FFT; PAMI entra como feature do residual.
detected_periods_by_sku = detect_seasonal_periods(
    benchmark_train,
    unique_id_col="unique_id",
    target_col="y",
    top_k=2,
)
pami_features = {}
for uid, group in benchmark_train.groupby("unique_id"):
    periods = [
        period for period in detected_periods_by_sku.get(uid, [])
        if period > 1 and period <= len(group) // 2
    ]
    detected_periods_by_sku[uid] = periods or [7]
    _, pami_features[uid] = first_pami_local_minimum(group["y"].to_numpy())

benchmark_train["pami_feature"] = benchmark_train["unique_id"].map(pami_features)
benchmark_test["pami_feature"] = benchmark_test["unique_id"].map(pami_features)


def callable_seasonal(period):
    return AutoETS(season_length=period, model="ZNA", alias=f"AutoETS-{period}")


def callable_trend():
    return AutoETS(model="ZAN", alias="Trend-AutoETS")


dmstl_pami = DMSTLWrapper(
    mf_resid=MLForecast(
        models=[RandomForestRegressor(n_estimators=120, random_state=42, n_jobs=-1)],
        freq="D",
        lags=[1, 7, 14, 28],
    ),
    season_length=detected_periods_by_sku,
    trend_model_callable=callable_trend,
    seasonal_model_callable=callable_seasonal,
)
dmstl_pami.fit(benchmark_train, target_col="y")
forecast_pami = dmstl_pami.predict(h=benchmark_horizon).rename(
    columns={"RandomForestRegressor": "forecast_pami"}
)
comparison_pami = benchmark_test.merge(
    forecast_pami[["unique_id", "ds", "forecast_pami"]],
    on=["unique_id", "ds"],
)
comparison_pami["forecast_pami"] = comparison_pami["forecast_pami"].clip(lower=0)
pami_aggregate = comparison_pami.groupby("ds")[["y", "forecast_pami"]].sum()
pami_metrics = pd.DataFrame(
    {
        "MAE": [mean_absolute_error(pami_aggregate["y"], pami_aggregate["forecast_pami"])],
        "RMSE": [mean_squared_error(pami_aggregate["y"], pami_aggregate["forecast_pami"]) ** 0.5],
        "actual_total": [pami_aggregate["y"].sum()],
        "forecast_total": [pami_aggregate["forecast_pami"].sum()],
    },
    index=["DMSTL_FFT_com_PAMI_feature"],
)
pami_metrics["total_error_pct"] = 100 * (
    pami_metrics["forecast_total"] / pami_metrics["actual_total"] - 1
)

print("Periodos sazonais detectados pelo FFT e feature PAMI por SKU:")
display(pd.DataFrame({"season_length": detected_periods_by_sku, "pami_feature": pami_features}))
print("Resultado agregado:")
display(pami_metrics)
ax = pami_aggregate.plot(figsize=(12, 5), marker="o")
ax.set_title("DMSTL: FFT para sazonalidade e PAMI como feature do residual")
ax.set_xlabel("Data")
ax.set_ylabel("Quantidade total dos SKUs")
ax.legend(["Realizado", "Forecast DMSTL"])
ax.grid(alpha=0.3)
plt.show()
pami_aggregate